# Layer 3: Exchange Timestamp Analysis (ME Profiling)
E1–E7: execution rate, micro-bursts, resolution, latency, out-of-order.

In [ ]:
import os, numpy as np, pandas as pd

DATA_DIR    = os.environ.get("DATA_DIR",    "/data/parquet")
REPORTS_DIR = os.environ.get("REPORTS_DIR", "/data/reports")
os.makedirs(REPORTS_DIR, exist_ok=True)

from mda.loader import load_trades
from mda.layer3.quantisation  import compute_timestamp_gaps, resolution_report
from mda.layer3.execution_rate import compute_execution_rate, execution_rate_stats
from mda.layer3.microbursts    import detect_microbursts, microburst_stats, microburst_heatmap
from mda.layer3.latency        import compute_feed_latency, latency_stats, latency_drift_timeseries
from mda.layer3.sequencing     import compute_out_of_order
from mda.plots.charts import (
    plot_timestamp_gap_hist, plot_execution_rate_ts,
    plot_microburst_size_hist, plot_microburst_heatmap,
    plot_latency_cdf, plot_latency_drift_ts,
    plot_oo_magnitude_hist, save_figure,
)
print("imports OK")

In [ ]:
df = load_trades(DATA_DIR)
print(f"Loaded {len(df):,} trades")

In [ ]:
gaps = compute_timestamp_gaps(df)
res  = resolution_report(df)
print("Timestamp resolution per exchange:")
print(res.to_string(index=False))

In [ ]:
fig = plot_timestamp_gap_hist(gaps)
fig.show()
save_figure(fig, "E4_timestamp_gap_hist", REPORTS_DIR)

In [ ]:
exec_rate  = compute_execution_rate(df)
exec_stats = execution_rate_stats(exec_rate)
print("Execution rate stats (exec/sec):")
print(exec_stats.to_string(index=False))

In [ ]:
fig = plot_execution_rate_ts(exec_rate)
fig.show()
save_figure(fig, "E1_execution_rate_ts", REPORTS_DIR)

In [ ]:
bursts   = detect_microbursts(df)
mb_stats = microburst_stats(bursts)
print("Micro-burst stats:")
print(mb_stats.to_string(index=False))

In [ ]:
fig = plot_microburst_size_hist(bursts)
fig.show()
save_figure(fig, "E2_microburst_size_hist", REPORTS_DIR)

In [ ]:
mb_heat = microburst_heatmap(bursts)
for exch in df["exchange"].unique():
    fig = plot_microburst_heatmap(mb_heat, exchange=exch)
    fig.show()
    save_figure(fig, f"E3_microburst_heatmap_{exch}", REPORTS_DIR)

In [ ]:
df = compute_feed_latency(df)
lat_stats = latency_stats(df)
print("Feed latency stats (ms):")
print(lat_stats.to_string(index=False))

In [ ]:
fig = plot_latency_cdf(df)
fig.show()
save_figure(fig, "E5_latency_cdf", REPORTS_DIR)

In [ ]:
drift = latency_drift_timeseries(df)
fig = plot_latency_drift_ts(drift)
fig.show()
save_figure(fig, "E6_latency_drift_ts", REPORTS_DIR)

In [ ]:
oo_summary, oo_events = compute_out_of_order(df)
print("Out-of-order summary:")
print(oo_summary.to_string(index=False))
fig = plot_oo_magnitude_hist(oo_events)
fig.show()
save_figure(fig, "E7_oo_magnitude_hist", REPORTS_DIR)